# Train preprocess: churn from DAC MVP

Ноутбук для запуска адаптированного `preprocess_train` под новую задачу: предсказать уход клиента из DAC в следующем месяце.

Логика сейчас такая:

- аудитория: клиенты, которые являются DAC в `base_month`;
- target: `target_churn_from_dac = 1`, если клиент не является DAC в следующем месяце;
- фичи: полный набор из текущего репозитория через `utils.load_features`;
- SQL берётся из `cvm_model.sql_my`.

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

import pandas as pd
pd.set_option('display.max_columns', None)

from cvm_model.io import State
from cvm_model.train.preprocess import preprocess_train
from cvm_model.parameters import features, input_suffix, target, template

## 1. Choose base month

`event_timestamp` трактуется как базовый месяц. Например, если поставить `2025-11-01`, аудитория будет DAC в ноябре, а таргет будет смотреть DAC-статус в декабре.

In [ ]:
event_timestamp = datetime(2025, 11, 1)
event_timestamp

## 2. Run preprocess

Эта ячейка очистит временный S3 `input`, соберёт датасет и сохранит parquet туда же, как обычный pipeline-step.

In [ ]:
preprocess_train(event_timestamp)

## 3. Read produced dataset

После успешного preprocess читаем parquet из временного S3 `input` и смотрим базовые проверки.

In [ ]:
state = State.from_env()
session = state.spark.session

input_prefix = state.settings.get_prefix(temp=True, suffix=input_suffix)
input_bucket = input_prefix.split('//')[1].split('/')[0]
input_key = '/'.join(input_prefix.split('//')[1].split('/')[1:])
input_path = template.format(bucket=input_bucket, prefix=input_key)

input_path

In [ ]:
df = session.read.parquet(input_path).toPandas()
df.shape

In [ ]:
display(df.head())
display(df.dtypes.to_frame('dtype'))

## 4. Checks

In [ ]:
display(df['target_churn_from_dac'].value_counts(dropna=False).to_frame('cnt'))
display(df['target_churn_from_dac'].value_counts(normalize=True, dropna=False).to_frame('share'))

if target in df.columns:
    display(df[target].value_counts(dropna=False).to_frame(f'{target}_cnt'))

In [ ]:
missing_features = sorted(set(features) - set(df.columns))
missing_features

In [ ]:
assert len(missing_features) == 0
assert df['contact_id'].nunique() == len(df)
assert df['target_churn_from_dac'].nunique() > 1, 'Таргет состоит из одного класса, нужна другая дата или проверка разметки.'

In [ ]:
null_report = df[features + ['target_churn_from_dac']].isna().mean().sort_values(ascending=False).to_frame('null_share')
display(null_report)
display(df[features].describe().T)